# Mini Transformer From Scratch

## What We Are Going To Build

This notebook is the "inside the machine" view of an LLM.
We will start with raw text, turn it into tokens, build a miniature decoder-only
transformer in PyTorch, train it on Shakespeare, inspect its attention patterns,
and generate new text from the trained model.

## Why This Notebook Exists

Large language models can feel mysterious because many teaching examples jump
straight from tokenized data to a giant pretrained checkpoint. Here we slow down
just enough to make the pipeline visible:

- raw text -> tokens
- tokens -> embeddings
- embeddings -> self-attention
- self-attention -> transformer blocks
- transformer blocks -> next-token logits
- logits -> softmax probabilities -> sampled text

## Learning Outcomes

By the end of the notebook, students should be able to explain:

- why tokenization is needed
- how embeddings and positional information work together
- why causal masking matters for GPT-style models
- what the attention equation is doing
- how the model is trained with cross-entropy loss
- what "memory" means in a fixed-context decoder
- how temperature and top-k sampling change generation

## Teaching Notes

- This notebook is designed to be visual and intuitive first, mathematical second.
- The attention and softmax math is explicitly shown, but we do not stay in derivations for long.
- There is checkpoint-aware logic later so class can load a saved model if full training is too slow.


# Environment Setup

This notebook installs the shared environment for the whole teaching sequence.
The repository bootstrap now handles PyTorch separately so NVIDIA Windows machines
can use a CUDA-enabled wheel instead of silently falling back to a CPU-only build.

In the code cell below we:

- locate the project root
- run the repository bootstrap script
- print progress so students can see the environment being prepared


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bootstrap_script = PROJECT_ROOT / "scripts" / "bootstrap_env.py"

print(f"Project root: {PROJECT_ROOT}")
print(f"Running bootstrap script: {bootstrap_script}")
subprocess.check_call([sys.executable, str(bootstrap_script)])
print("Environment ready. If PyTorch was replaced during bootstrap, restart the kernel once before continuing.")


## Imports, Plotting, and Reproducibility

Before we touch the model, we set up the tools we will reuse throughout the notebook.
The theory idea is simple: experiments are easier to teach when they are reproducible
and when plots use a consistent visual style.

In the code cell below we:

- import PyTorch, plotting, and dataset utilities
- set random seeds
- detect whether a GPU is available
- create the artifact folders used for checkpoints


In [ ]:
import math
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from datasets import load_dataset
from IPython.display import clear_output, display
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"Detected GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not detected. The notebook still runs, but training will be slower.")


## Notebook Configuration

This notebook supports two modes:

- a faster classroom walkthrough
- a longer side-project training run

The theory idea is that teaching notebooks should not be hostage to wall-clock time.
We want one set of cells that can either train a small model live or load a saved checkpoint.

In the code cell below we:

- choose the active preset
- define the model size and training schedule
- point to the checkpoint location used later
- display the active choices in a table


In [ ]:
RUN_FULL_SIDE_PROJECT = False
USE_EXISTING_CHECKPOINT_IF_AVAILABLE = True

PRESETS = {
    "walkthrough": {
        "batch_size": 32,
        "block_size": 128,
        "n_embed": 192,
        "n_head": 6,
        "n_layer": 4,
        "dropout": 0.2,
        "learning_rate": 3e-4,
        "max_iters": 500,
        "eval_interval": 50,
        "eval_batches": 25,
    },
    "side_project": {
        "batch_size": 48,
        "block_size": 160,
        "n_embed": 256,
        "n_head": 8,
        "n_layer": 6,
        "dropout": 0.2,
        "learning_rate": 2e-4,
        "max_iters": 1800,
        "eval_interval": 100,
        "eval_batches": 40,
    },
}

ACTIVE_PRESET = "side_project" if RUN_FULL_SIDE_PROJECT else "walkthrough"
CONFIG = PRESETS[ACTIVE_PRESET].copy()
CHECKPOINT_PATH = CHECKPOINT_DIR / f"mini_transformer_{ACTIVE_PRESET}.pt"

config_df = pd.DataFrame(
    [{"setting": key, "value": value} for key, value in CONFIG.items()]
)
display(config_df)
print(f"Checkpoint path: {CHECKPOINT_PATH}")


## Step 1: Load the Raw Dataset

Language models start with text, not tensors.
Before we tokenize anything, it is useful to inspect the raw corpus so students can see
what kind of language distribution the model will learn from.

In the code cell below we:

- load `karpathy/tiny_shakespeare` from Hugging Face
- collapse the dataset into one training string
- print a preview of the corpus
- visualize the most frequent characters and the line-length distribution


In [ ]:
dataset = load_dataset("karpathy/tiny_shakespeare")
split_name = "train" if "train" in dataset else list(dataset.keys())[0]
text_column = "text" if "text" in dataset[split_name].column_names else dataset[split_name].column_names[0]

text_rows = dataset[split_name][text_column]
if isinstance(text_rows, str):
    raw_text = text_rows
elif len(text_rows) == 1:
    raw_text = text_rows[0]
else:
    raw_text = "\n".join(text_rows)

print(f"Loaded split: {split_name}")
print(f"Characters in corpus: {len(raw_text):,}")
print()
print(raw_text[:900])

char_counts = Counter(raw_text)
frequency_df = (
    pd.DataFrame(char_counts.items(), columns=["character", "count"])
    .sort_values("count", ascending=False)
    .head(20)
)
line_lengths = pd.Series([len(line) for line in raw_text.splitlines() if line.strip()])

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sns.barplot(data=frequency_df, x="count", y="character", palette="viridis", ax=axes[0])
axes[0].set_title("Most frequent characters")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Character")

sns.histplot(line_lengths, bins=30, color="teal", ax=axes[1])
axes[1].set_title("Distribution of non-empty line lengths")
axes[1].set_xlabel("Characters per line")
axes[1].set_ylabel("Number of lines")
plt.tight_layout()
plt.show()


## Step 2: Build a Tokenizer From Scratch

A model cannot consume raw Python strings directly.
We need a mapping from symbols to integers.

For a first-principles teaching notebook, a character-level tokenizer is ideal because:

- every token is easy to inspect
- the vocabulary is tiny
- students can directly see how text becomes integers

In the code cell below we:

- create a character vocabulary
- build `stoi` and `itos` lookup tables
- define encode and decode functions
- visualize a short text snippet and its token ids


In [ ]:
vocabulary = sorted(set(raw_text))
vocab_size = len(vocabulary)
stoi = {ch: idx for idx, ch in enumerate(vocabulary)}
itos = {idx: ch for ch, idx in stoi.items()}

def encode(text):
    return [stoi[ch] for ch in text]

def decode(token_ids):
    return "".join(itos[idx] for idx in token_ids)

encoded_text = torch.tensor(encode(raw_text), dtype=torch.long)
sample_text = "ROMEO:"
sample_tokens = encode(sample_text)

print(f"Vocabulary size: {vocab_size}")
display(
    pd.DataFrame(
        {
            "character": list(sample_text),
            "token_id": sample_tokens,
        }
    )
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
token_stream = encoded_text[:80].cpu().numpy()
axes[0].plot(token_stream, marker="o", linewidth=2, color="slateblue")
axes[0].set_title("First 80 token ids in the corpus")
axes[0].set_xlabel("Token position")
axes[0].set_ylabel("Token id")

sns.heatmap(
    np.array(sample_tokens).reshape(1, -1),
    annot=np.array(list(sample_text)).reshape(1, -1),
    fmt="",
    cmap="magma",
    cbar=False,
    ax=axes[1],
)
axes[1].set_title("Character -> token id mapping for a sample word")
axes[1].set_xlabel("Character position")
axes[1].set_yticks([])
plt.tight_layout()
plt.show()


## Step 3: Create Training and Validation Splits

Training a next-token predictor means teaching the model to map a context
to the token that comes one step later.

So instead of using labels from a separate file, we create labels by shifting the text.
This is the core language-modeling trick: the next token is the target.

In the code cell below we:

- split the integer token stream into train and validation regions
- build a small context/target example
- visualize how each input position predicts the next token


In [ ]:
split_index = int(0.9 * len(encoded_text))
train_data = encoded_text[:split_index]
val_data = encoded_text[split_index:]

demo_block_size = 24
demo_context = train_data[:demo_block_size]
demo_target = train_data[1 : demo_block_size + 1]

shift_df = pd.DataFrame(
    {
        "position": list(range(demo_block_size)),
        "input_char": [itos[int(token)] if itos[int(token)] != "\n" else "\\n" for token in demo_context],
        "input_id": demo_context.tolist(),
        "target_char": [itos[int(token)] if itos[int(token)] != "\n" else "\\n" for token in demo_target],
        "target_id": demo_target.tolist(),
    }
)

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")
display(shift_df)

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(
    np.vstack([demo_context.cpu().numpy(), demo_target.cpu().numpy()]),
    cmap="crest",
    cbar=True,
    yticklabels=["input ids", "shifted targets"],
    xticklabels=list(range(demo_block_size)),
    ax=ax,
)
ax.set_title("Language modeling target = input shifted by one step")
ax.set_xlabel("Token position inside one training chunk")
plt.tight_layout()
plt.show()


## Step 4: Sample Random Batches

Transformers do not train on one giant uninterrupted stream at a time.
We randomly crop many short context windows so the model sees a diverse mix of positions.

In the code cell below we:

- define a `get_batch` function
- sample a random batch from the training split
- visualize the token-id grid the model will see during one optimization step


In [ ]:
def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - CONFIG["block_size"] - 1, (CONFIG["batch_size"],))
    x = torch.stack([source[start : start + CONFIG["block_size"]] for start in starts])
    y = torch.stack([source[start + 1 : start + CONFIG["block_size"] + 1] for start in starts])
    return x.to(DEVICE), y.to(DEVICE)

xb, yb = get_batch("train")
print(f"Batch shape for inputs: {tuple(xb.shape)}")
print(f"Batch shape for targets: {tuple(yb.shape)}")

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(xb[:10].cpu().numpy(), cmap="rocket", ax=ax)
ax.set_title("Token ids in a random training batch")
ax.set_xlabel("Position inside the context window")
ax.set_ylabel("Batch row")
plt.tight_layout()
plt.show()


## Step 5: Turn Token Ids Into Embeddings

Token ids are categorical labels, not continuous representations.
The embedding layer learns a dense vector for each token, and positional embeddings tell the
model where each token lives inside the sequence.

In the code cell below we:

- define a tiny embedding module
- look at token and position embeddings separately
- visualize the first few embedding dimensions as heatmaps


In [ ]:
class TokenPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, block_size, n_embed):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)

    def forward(self, idx):
        batch_size, time_steps = idx.shape
        token_vectors = self.token_embedding(idx)
        positions = torch.arange(time_steps, device=idx.device)
        position_vectors = self.position_embedding(positions)
        return token_vectors, position_vectors

embedding_preview = TokenPositionEmbedding(vocab_size, CONFIG["block_size"], CONFIG["n_embed"]).to(DEVICE)
token_vectors, position_vectors = embedding_preview(xb[:1])
combined_vectors = token_vectors + position_vectors.unsqueeze(0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(token_vectors[0, :12, :24].detach().cpu(), cmap="coolwarm", ax=axes[0])
axes[0].set_title("Token embeddings")
axes[0].set_xlabel("Embedding dimension")
axes[0].set_ylabel("Token position")

sns.heatmap(position_vectors[:12, :24].detach().cpu(), cmap="crest", ax=axes[1])
axes[1].set_title("Position embeddings")
axes[1].set_xlabel("Embedding dimension")
axes[1].set_ylabel("Sequence position")

sns.heatmap(combined_vectors[0, :12, :24].detach().cpu(), cmap="magma", ax=axes[2])
axes[2].set_title("Combined signal sent into attention")
axes[2].set_xlabel("Embedding dimension")
axes[2].set_ylabel("Token position")
plt.tight_layout()
plt.show()


## Step 6: Inspect the Attention Equation

The core attention equation is:

`Attention(Q, K, V) = softmax(QK^T / sqrt(d_k) + mask) V`

Intuitively:

- queries ask: "what information am I looking for?"
- keys say: "what kind of information do I contain?"
- values carry the actual content to mix together
- softmax turns raw compatibility scores into probabilities
- the causal mask prevents looking into the future

In the code cell below we:

- create one attention head by hand
- compute raw scores, masked scores, and attention probabilities
- visualize the causal mask and the final weighted lookup


In [ ]:
attention_demo_length = 12
demo_x = combined_vectors[:, :attention_demo_length, :]

query_layer = nn.Linear(CONFIG["n_embed"], CONFIG["n_embed"], bias=False).to(DEVICE)
key_layer = nn.Linear(CONFIG["n_embed"], CONFIG["n_embed"], bias=False).to(DEVICE)
value_layer = nn.Linear(CONFIG["n_embed"], CONFIG["n_embed"], bias=False).to(DEVICE)

queries = query_layer(demo_x)
keys = key_layer(demo_x)
values = value_layer(demo_x)

raw_scores = queries @ keys.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(CONFIG["n_embed"])
causal_mask = torch.tril(torch.ones(attention_demo_length, attention_demo_length, device=DEVICE))
masked_scores = scaled_scores.masked_fill(causal_mask == 0, float("-inf"))
attention_weights = torch.softmax(masked_scores, dim=-1)
context_vectors = attention_weights @ values

plot_tokens = [
    itos[int(token)] if itos[int(token)] != "\n" else "\\n"
    for token in xb[0, :attention_demo_length].cpu()
]

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
sns.heatmap(raw_scores[0].detach().cpu(), cmap="icefire", ax=axes[0])
axes[0].set_title("Raw QK^T scores")
axes[0].set_xlabel("Key positions")
axes[0].set_ylabel("Query positions")

sns.heatmap(causal_mask.detach().cpu(), cmap="Greys", cbar=False, ax=axes[1])
axes[1].set_title("Causal mask")
axes[1].set_xlabel("Visible key positions")
axes[1].set_ylabel("Query positions")

sns.heatmap(
    attention_weights[0].detach().cpu(),
    cmap="viridis",
    xticklabels=plot_tokens,
    yticklabels=plot_tokens,
    ax=axes[2],
)
axes[2].set_title("Softmax attention weights")
axes[2].set_xlabel("Keys")
axes[2].set_ylabel("Queries")

sns.heatmap(context_vectors[0, :, :24].detach().cpu(), cmap="mako", ax=axes[3])
axes[3].set_title("Weighted value vectors")
axes[3].set_xlabel("Embedding dimension")
axes[3].set_ylabel("Token position")
plt.tight_layout()
plt.show()

row_sums = attention_weights[0].sum(dim=-1).detach().cpu().numpy()
print("Each attention row sums to:", np.round(row_sums, 3))


## Step 7: Build the Mini GPT Model

Now we assemble the actual network.
A decoder-only transformer block repeats three ideas:

- layer normalization to stabilize training
- masked multi-head self-attention to mix information across positions
- a feed-forward network to transform each position independently

In the code cell below we:

- define multi-head self-attention
- define a transformer block
- define the full decoder-only language model
- keep the code commented so students can map every line to the theory


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        assert n_embed % n_head == 0, "Embedding dimension must divide evenly across heads."
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=False)
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x, return_attention=False):
        batch_size, time_steps, channels = x.shape
        qkv = self.qkv(x)
        qkv = qkv.view(batch_size, time_steps, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv[0], qkv[1], qkv[2]

        scores = (queries @ keys.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:time_steps, :time_steps] == 0, float("-inf"))
        attention = torch.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        mixed = attention @ values
        mixed = mixed.transpose(1, 2).contiguous().view(batch_size, time_steps, channels)
        output = self.proj(mixed)
        output = self.dropout(output)

        if return_attention:
            return output, attention
        return output


class FeedForward(nn.Module):
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
        self.attn = MultiHeadSelfAttention(n_embed, n_head, block_size, dropout)
        self.ff = FeedForward(n_embed, dropout)

    def forward(self, x, return_attention=False):
        if return_attention:
            attn_output, attention = self.attn(self.ln1(x), return_attention=True)
            x = x + attn_output
            x = x + self.ff(self.ln2(x))
            return x, attention

        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embed, n_head, n_layer, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)
        self.blocks = nn.ModuleList(
            [Block(n_embed, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.final_norm = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None, return_attentions=False):
        batch_size, time_steps = idx.shape
        positions = torch.arange(time_steps, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)

        first_block_attention = None
        for block_index, block in enumerate(self.blocks):
            if return_attentions and block_index == 0:
                x, first_block_attention = block(x, return_attention=True)
            else:
                x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            batch_size, time_steps, vocab_size = logits.shape
            loss = nn.functional.cross_entropy(
                logits.view(batch_size * time_steps, vocab_size),
                targets.view(batch_size * time_steps),
            )

        if return_attentions:
            return logits, loss, first_block_attention
        return logits, loss


## Step 8: Instantiate the Model and Inspect Its Size

It is useful to check model size before training so students can connect architecture choices
to parameter count, memory cost, and runtime.

In the code cell below we:

- instantiate the mini GPT model
- run one forward pass on a random batch
- count the parameters by top-level component
- visualize where the learnable capacity lives


In [ ]:
model = MiniGPT(
    vocab_size=vocab_size,
    block_size=CONFIG["block_size"],
    n_embed=CONFIG["n_embed"],
    n_head=CONFIG["n_head"],
    n_layer=CONFIG["n_layer"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

test_logits, test_loss = model(xb, yb)
print(f"Logits shape: {tuple(test_logits.shape)}")
print(f"Initial random-weight loss: {test_loss.item():.4f}")

component_totals = {}
for name, parameter in model.named_parameters():
    component_name = name.split(".")[0]
    component_totals.setdefault(component_name, 0)
    component_totals[component_name] += parameter.numel()

parameter_df = pd.DataFrame(
    [{"component": key, "parameters": value} for key, value in component_totals.items()]
).sort_values("parameters", ascending=False)

total_parameters = int(parameter_df["parameters"].sum())
print(f"Total parameters: {total_parameters:,}")
display(parameter_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=parameter_df, x="parameters", y="component", palette="flare")
plt.title("Where the model parameters live")
plt.xlabel("Number of parameters")
plt.ylabel("Component")
plt.tight_layout()
plt.show()


## Step 9: Train or Load a Checkpoint

The model learns by comparing its predicted next-token distribution to the true next token,
then using gradient descent to reduce the cross-entropy loss.

In the code cell below we:

- create an optimizer
- define a small validation routine
- either load an existing checkpoint or run training
- visualize the training and validation loss over time


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        split_losses = []
        for _ in range(CONFIG["eval_batches"]):
            batch_x, batch_y = get_batch(split)
            _, loss = model(batch_x, batch_y)
            split_losses.append(loss.item())
        losses[split] = float(np.mean(split_losses))
    model.train()
    return losses

history = []

if CHECKPOINT_PATH.exists() and USE_EXISTING_CHECKPOINT_IF_AVAILABLE:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    history = checkpoint.get("history", [])
    print(f"Loaded checkpoint from {CHECKPOINT_PATH}")
else:
    print("Starting training from scratch...")
    model.train()
    for step in tqdm(range(CONFIG["max_iters"]), desc="Training mini GPT"):
        batch_x, batch_y = get_batch("train")
        _, loss = model(batch_x, batch_y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        should_log = step % CONFIG["eval_interval"] == 0 or step == CONFIG["max_iters"] - 1
        if should_log:
            loss_snapshot = estimate_loss()
            history.append(
                {
                    "step": step,
                    "train_loss": loss_snapshot["train"],
                    "val_loss": loss_snapshot["val"],
                }
            )

            clear_output(wait=True)
            history_df = pd.DataFrame(history)
            print(
                f"Step {step:>4} | train loss {loss_snapshot['train']:.4f} | "
                f"val loss {loss_snapshot['val']:.4f}"
            )
            plt.figure(figsize=(10, 5))
            plt.plot(history_df["step"], history_df["train_loss"], label="train")
            plt.plot(history_df["step"], history_df["val_loss"], label="validation")
            plt.title("Learning curve for the from-scratch transformer")
            plt.xlabel("Training step")
            plt.ylabel("Cross-entropy loss")
            plt.legend()
            plt.tight_layout()
            plt.show()

    torch.save(
        {
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": CONFIG,
            "history": history,
        },
        CHECKPOINT_PATH,
    )
    print(f"Saved checkpoint to {CHECKPOINT_PATH}")

if history:
    history_df = pd.DataFrame(history)
    plt.figure(figsize=(10, 5))
    plt.plot(history_df["step"], history_df["train_loss"], label="train")
    plt.plot(history_df["step"], history_df["val_loss"], label="validation")
    plt.title("Final loss curves")
    plt.xlabel("Training step")
    plt.ylabel("Cross-entropy loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Step 10: Visualize Attention and LLM Memory

When people say an LLM "remembers" context, they do not mean memory in the human sense.
In a decoder-only transformer, memory means:

- the model only sees tokens inside the current context window
- each token attends back to earlier visible tokens
- the attention weights decide which earlier positions matter most

In the code cell below we:

- run the model on one context window
- visualize one block's average attention map
- inspect what the last token is attending to


In [ ]:
model.eval()

attention_slice = 24
context_window = train_data[: CONFIG["block_size"]].unsqueeze(0).to(DEVICE)
_, _, attention_map = model(context_window, return_attentions=True)

average_attention = attention_map[0].mean(dim=0).detach().cpu().numpy()
display_tokens = [
    itos[int(token)] if itos[int(token)] != "\n" else "\\n"
    for token in context_window[0, :attention_slice].cpu()
]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(
    average_attention[:attention_slice, :attention_slice],
    cmap="viridis",
    xticklabels=display_tokens,
    yticklabels=display_tokens,
    ax=axes[0],
)
axes[0].set_title("Average attention across heads in the first transformer block")
axes[0].set_xlabel("Keys: earlier tokens the model can look at")
axes[0].set_ylabel("Queries: current token positions")

last_token_attention = average_attention[attention_slice - 1, :attention_slice]
axes[1].bar(range(attention_slice), last_token_attention, color="darkorange")
axes[1].set_title("Where the last visible token puts its attention mass")
axes[1].set_xlabel("Earlier position")
axes[1].set_ylabel("Attention weight")
axes[1].set_xticks(range(attention_slice))
axes[1].set_xticklabels(display_tokens, rotation=90)
plt.tight_layout()
plt.show()


## Step 11: Generate Text With Different Decoding Strategies

The model outputs logits, not words.
To turn logits into generated text we:

- convert logits to probabilities with softmax
- optionally scale them with temperature
- optionally keep only the top-k candidates
- sample or take the argmax

In the code cell below we:

- define a small text-generation helper
- compare multiple decoding settings from the same prompt
- display the resulting text so students can see how randomness changes behavior


In [ ]:
@torch.no_grad()
def generate_text(prompt, max_new_tokens=140, temperature=1.0, top_k=None):
    model.eval()
    running_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)

    for _ in range(max_new_tokens):
        idx_cond = running_ids[:, -CONFIG["block_size"] :]
        logits, _ = model(idx_cond)
        next_token_logits = logits[:, -1, :]

        if temperature == 0:
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        else:
            next_token_logits = next_token_logits / temperature
            if top_k is not None:
                top_values, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
                cutoff = top_values[:, [-1]]
                next_token_logits = next_token_logits.masked_fill(next_token_logits < cutoff, float("-inf"))
            probabilities = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)

        running_ids = torch.cat([running_ids, next_token], dim=1)

    return decode(running_ids[0].tolist())

generation_prompt = "ROMEO:\n"
generations = [
    {
        "strategy": "Greedy-like (temperature 0)",
        "text": generate_text(generation_prompt, temperature=0, top_k=None),
    },
    {
        "strategy": "Warm sampling (temperature 0.8, top_k 20)",
        "text": generate_text(generation_prompt, temperature=0.8, top_k=20),
    },
    {
        "strategy": "Looser sampling (temperature 1.1, top_k 40)",
        "text": generate_text(generation_prompt, temperature=1.1, top_k=40),
    },
]

generation_df = pd.DataFrame(generations)
display(generation_df)


## Step 12: Visualize the Context Window as a Memory Budget

A decoder-only transformer does not have unlimited memory.
It can only directly use the last `block_size` tokens.

In the code cell below we:

- simulate generation over time
- show how many tokens are visible at each step
- draw a visibility map that makes the fixed memory budget concrete


In [ ]:
prompt_ids = encode("ROMEO:\n")
simulated_steps = 80

visible_history = [min(len(prompt_ids) + step, CONFIG["block_size"]) for step in range(simulated_steps)]
visibility = np.zeros((simulated_steps, len(prompt_ids) + simulated_steps))

for step in range(simulated_steps):
    total_length = len(prompt_ids) + step
    visible_start = max(0, total_length - CONFIG["block_size"])
    visibility[step, visible_start:total_length] = 1

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].plot(visible_history, linewidth=3, color="forestgreen")
axes[0].axhline(CONFIG["block_size"], linestyle="--", color="firebrick", label="context limit")
axes[0].set_title("How much history the model can see while generating")
axes[0].set_xlabel("Generation step")
axes[0].set_ylabel("Visible tokens")
axes[0].legend()

sns.heatmap(
    visibility[:, :120],
    cmap="Greens",
    cbar=False,
    ax=axes[1],
)
axes[1].set_title("Memory visibility map")
axes[1].set_xlabel("Token position in the growing sequence")
axes[1].set_ylabel("Generation step")
plt.tight_layout()
plt.show()


## Wrap-Up

This notebook built a real miniature transformer from raw text to generated output.
The model is tiny compared with production LLMs, but the pipeline is the same:

- represent text as tokens
- learn embeddings
- mix information with masked self-attention
- train with next-token prediction
- decode with sampling

The next notebook extends this story to encoder-decoder models and sequence-to-sequence tasks.
